# CAMP 1: Learning registration

Standalone version of practical section 5. Run locally from the practical folder, or upload this notebook to Google Colab and upload the student ZIP when prompted. Use a CPU runtime. Work one step at a time; inspect each result before continuing.

In [ ]:
# Install missing registration dependencies; on Colab upload the student ZIP once.
import sys
from pathlib import Path
try:
    from google.colab import files as colab_files
    on_colab = True
except ImportError:
    on_colab = False
if not on_colab:
    import importlib.util
    import subprocess
    if importlib.util.find_spec("torch") is None:
        print("Installing PyTorch into the notebook's Python environment...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch>=2.2,<3"])
        importlib.invalidate_caches()
# SimpleITK is needed locally and on Colab for Mattes mutual information.
import importlib.util
import subprocess
if importlib.util.find_spec("SimpleITK") is None:
    print("Installing SimpleITK for Mattes mutual information...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "SimpleITK>=2.5,<3"])
    importlib.invalidate_caches()
if on_colab and not (Path.cwd()/"registration_practical"/"registration_lab.py").exists() and not (Path.cwd()/"registration_lab.py").exists():
    import io, zipfile
    print("Upload CAMP1_registration_practical_student.zip (includes helper and CT crops).")
    uploaded = colab_files.upload()
    archive_name = next(name for name in uploaded if name.endswith(".zip"))
    with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as archive:
        root = Path.cwd().resolve()
        for name in archive.namelist():
            if not (root/name).resolve().is_relative_to(root):
                raise ValueError("Unexpected path in archive.")
        archive.extractall(root)
    del uploaded


## 5. Learning registration from image agreement

The question for this section is: **how does comparing a warped image with a fixed image teach a network to predict a displacement field?**

A small encoder/decoder receives two CT images and predicts two offset components on a **5×5 control grid**. Bicubic interpolation expands this coarse prediction to a dense sampling field, followed by smooth bounding to ±12 pixels. This prevents the model from fitting independent pixel-scale ripples; it is not a cubic B-spline or a biomechanical model. A differentiable sampler warps the moving image. Image mismatch and a smoothness penalty train the shared network weights. This illustrates the VoxelMorph principle with a smaller teaching architecture, not a reproduction of the published model.

The images are real CT crops at their native 128×128 resolution, with no downsampling. Broad regional motion is generated from a **3×3 control grid** plus translation, with smooth bounding and known backward displacements. The amplitude is 12 pixels per component before smooth bounding (2.4 times the earlier example). Control nodes span the image, so the motion varies over large regions instead of following individual image edges. The known field is used only for evaluation, never for training. Training and evaluation source slices are disjoint but come from one volume; this is not an evaluation on independent patients. Both splits use the same motion generator. Distances here are source-grid pixels (2 mm per pixel).

Run each step and inspect it before continuing. The network is initialized from scratch and trained on CPU in this notebook. For local setup, install the requirements and run the first setup cell; it automatically installs PyTorch into the notebook's Python environment if missing. In Colab, use the default CPU runtime and upload the student ZIP in the setup cell above. No Google Drive mount is needed.

### 5.1 Run the network before training

Run this setup once. It resets the learning model and log and creates a new image pair. After updating the helper files, rerun this step: reloading a module alone does not replace an existing image pair or network. Check the printed version and displacement below. The last layer starts near zero, so the initial displacement is close to identity. Inspect the target, moving and registered images, both overlays, the sampling grid and Jacobian. The before/after difference maps show image minus target with the same signed color scale, anchored to the unregistered pair. Gray areas are outside the evaluation ROI or unavailable. Compare the initial overlay with the network output.

In [ ]:
from pathlib import Path
import sys, importlib, copy
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
try:
    import torch
except ImportError as exc:
    raise ImportError('Install PyTorch with %pip install "torch>=2.2,<3", then rerun this cell.') from exc

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd()/"registration_practical",
              Path.cwd()/"new camp exercise"/"registration_practical"]
DL_DIR = next((p for p in candidates if (p/"registration_learning.py").is_file()), None)
if DL_DIR is None:
    raise FileNotFoundError("Keep the complete updated practical folder together; in Colab upload its ZIP first.")
sys.path.insert(0, str(DL_DIR))
import registration_lab as lab
lab = importlib.reload(lab)
import registration_learning as dl
dl = importlib.reload(dl)
expected_version = "coarse5x5_motion12_v2"
if getattr(dl, "ARCHITECTURE_VERSION", None) != expected_version:
    raise RuntimeError(f"Outdated helper at {dl.__file__}. Replace the complete practical folder with the updated student ZIP; on Colab extract the new ZIP or start a fresh runtime.")
torch.set_num_threads(2)

def show_learning(fig):
    display(fig)
    plt.close(fig)

def learning_log(rows):
    lines = ["| Stage | Updates | Image MSE | R | Field error [px] | Nonpositive Jacobians [%] |",
             "| --- | --- | --- | --- | --- | --- |"]
    for name, updates, result in rows:
        lines.append(f"| {name} | {updates} | {result['image']:.6f} | {result['regularity']:.5f} | {result['field_error']:.3f} | {result['folds']:.2f} |")
    display(Markdown("\n".join(lines)))

network = dl.new_model()  # Fresh weights; no checkpoint is loaded.
optimizer = torch.optim.Adam(network.parameters(), lr=dl.LEARNING_RATE)
learning_updates = 0
learning_history = []
learning_trials = []
training_example = dl.make_batch("train", seed=42, index=6)
untrained_result = dl.evaluate(network, training_example)
learning_trials.append(("Before training", learning_updates, untrained_result))
motion = training_example["truth"].norm(dim=1)[:, dl.MARGIN:-dl.MARGIN, dl.MARGIN:-dl.MARGIN]
print("Learning version:", dl.ARCHITECTURE_VERSION)
print("Helper:", Path(dl.__file__).resolve())
print(f"Generated displacement in loss ROI: mean {motion.mean().item():.2f} px; maximum {motion.max().item():.2f} px")
print("Device: CPU | Parameters:", sum(p.numel() for p in network.parameters()), "| PyTorch:", torch.__version__)
show_learning(dl.plot_result(training_example, untrained_result, "Before training"))
learning_log(learning_trials)

### 5.2 Compute the training loss

The objective is `E = S + lambda * R`. `S` is image MSE in an interior region. `R` averages squared neighboring differences of the predicted displacement. The weight is 0.01 in the training loop below. The coarse output representation also limits spatial detail; this is separate from the regularization penalty. The field is bounded to ±12 pixels; a sixteen-pixel margin excludes padding from the image loss.

Read the arguments to the loss below: no ground-truth field is passed. Compare the two contributions. Is a small smoothness penalty by itself evidence of a useful registration?

In [ ]:
network.eval()
with torch.no_grad():
    predicted_field = network(training_example["fixed"], training_example["moving"])
    warped_image = dl.warp(training_example["moving"], predicted_field)
    total_loss, image_loss, regularity = dl.losses(
        training_example["fixed"], training_example["moving"], predicted_field)
print(f"Image mismatch S = {image_loss.item():.6f}")
print(f"Regularity R = {regularity.item():.6f}; weighted contribution = {dl.WEIGHT*regularity.item():.6f}")
show_learning(dl.plot_loss_parts(dl.evaluate(network, training_example)))

### 5.3 Follow one training update

Predict what changes: the input images, the field values, or the shared network weights? Run the update below. It shows the image-loss gradient with respect to displacement and checks that network weights changed. The images and known evaluation field stay fixed. Rerunning this cell performs another update on this same pair; the following training block uses many generated pairs.

In [ ]:
network.train()
optimizer.zero_grad(set_to_none=True)
weights_before = torch.cat([p.detach().flatten().clone() for p in network.parameters()])
predicted_field = network(training_example["fixed"], training_example["moving"])
total_loss, image_loss, regularity = dl.losses(
    training_example["fixed"], training_example["moving"], predicted_field)
# Inspect the learning signal passed through the sampler, before updating weights.
image_gradient = torch.autograd.grad(image_loss, predicted_field, retain_graph=True)[0]
total_loss.backward()
network_gradient = torch.sqrt(sum(p.grad.square().sum() for p in network.parameters() if p.grad is not None))
optimizer.step()
learning_updates += 1
weights_after = torch.cat([p.detach().flatten() for p in network.parameters()])
print("Network gradient norm:", network_gradient.item())
print("Weight-change norm:", torch.linalg.vector_norm(weights_after-weights_before).item())
show_learning(dl.plot_gradient(training_example, {"field_gradient": image_gradient[0].norm(dim=0).detach().numpy()}))
one_update_result = dl.evaluate(network, training_example)
learning_trials.append(("Live weight update", learning_updates, one_update_result))
show_learning(dl.plot_result(training_example, one_update_result, "After one weight update"))
learning_log(learning_trials)

### 5.4 Train for 1,000 additional updates

Continue training the network from the single update above. This cell performs **1,000 additional optimizer updates**, using batches of eight newly generated training pairs. Starting at step 5.1 and running step 5.3 once gives **1,001 updates in total**. The same optimizer and weights continue throughout; no saved model is loaded.

Write a prediction, then run the training cell. Progress and batch loss are printed every 100 updates. Compare the trained output with the earlier stages on the same displayed example. Rerunning the training cell adds another 1,000 updates; rerun step 5.1 to start over.

In [ ]:
prediction = ""  # Predict how the alignment and field will change.

In [ ]:
training_steps = 1000
for block_step in range(1, training_steps + 1):
    batch = dl.make_batch("train", count=8, seed=1000 + learning_updates)
    step_result = dl.train_update(network, optimizer, batch)
    learning_updates += 1
    learning_history.append(dict(step=learning_updates, **step_result))
    if block_step % 100 == 0:
        print(f"Training {block_step}/{training_steps} | total updates {learning_updates} | "
              f"batch loss {step_result['total']:.6f}", flush=True)

progress_result = dl.evaluate(network, training_example)
learning_trials.append(("After full live training", learning_updates, progress_result))
print("Prediction:", prediction or "Not recorded")
show_learning(dl.plot_result(training_example, progress_result, f"Trained here: {learning_updates} updates"))
learning_log(learning_trials)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([r["step"] for r in learning_history], [r["total"] for r in learning_history])
ax.set(xlabel="Shared-weight updates", ylabel="Training batch loss S + λR")
show_learning(fig)

### 5.5 Run inference with frozen weights

Choose a held-out pair. Its CT source slice was excluded from training, and its motion follows the same generator and range as the training pairs. Run one forward pass with frozen network weights. No optimizer step is performed. The predicted field can differ across inputs while the weights remain unchanged.

This uses the network you just trained in step 5.4. To inspect another held-out pair, change `test_slice` or `test_seed` and rerun this step.

In [ ]:
test_slice = 0  # 0, 1, 2 or 3 in the held-out slice set.
test_seed = 84
inference_network = copy.deepcopy(network).eval()
for parameter in inference_network.parameters():
    parameter.requires_grad_(False)
inference_network.zero_grad(set_to_none=True)
held_out_pair = dl.make_batch("test", seed=test_seed, index=test_slice)
frozen_before = {k: v.clone() for k, v in inference_network.state_dict().items()}
inference_result = dl.evaluate(inference_network, held_out_pair)
weights_unchanged = all(torch.equal(frozen_before[k], v) for k, v in inference_network.state_dict().items())
print("Weights unchanged:", weights_unchanged)
print("Inference field requires no optimizer updates.")
show_learning(dl.plot_result(held_out_pair, inference_result, "Held-out pair: frozen-weight inference"))

**Discuss:** trace the path from image mismatch to sampler coordinates to network weights. Which operations are needed at inference, and which are only needed for training?

References: [VoxelMorph](https://arxiv.org/abs/1809.05231), [PyTorch grid_sample](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html). Dataset preparation and deformation construction are in `data/README.md`.